# ⚙️ api-forge
### *Turn any API docs into a working wrapper class — in under 30 seconds*

---

Every new API integration starts the same way: open the docs, figure out the base URL, figure out auth, write a thin `requests` wrapper you'll copy-paste into every project forever. **api-forge automates that first hour of boilerplate.**

This notebook walks through the full pipeline on two real APIs:
- **JSONPlaceholder** — no spec file, so the LLM reads the prose docs
- **Petstore (Swagger)** — has an OpenAPI spec, so we skip the LLM entirely

Both produce the same output shape. The rest of the pipeline never needs to know which path ran.

> 📌 **GitHub:** https://github.com/ajsalsameer/api-forge  
> 🎯 **Hackathon:** Claysys — Smart DevTool for API Integration

---
## 0 · Setup

Install dependencies and clone the repo. If you're re-running this notebook, the `if not os.path.exists` guard prevents a second clone.

In [ ]:
!pip install requests beautifulsoup4 groq python-dotenv lxml streamlit -q
print("Dependencies installed.")

In [ ]:
import os, sys, json, textwrap
from IPython.display import Markdown, display

if not os.path.exists("api-forge"):
    !git clone https://github.com/ajsalsameer/api-forge.git
    print("Cloned.")
else:
    # Pull latest changes if the repo already exists
    !cd api-forge && git pull --quiet
    print("Repo already present — pulled latest.")

os.chdir("api-forge")
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

## 1 · Groq API key

api-forge uses [Groq](https://console.groq.com) for LLM calls — it's free, no card needed, and fast enough that extraction usually finishes in under 3 seconds.

`getpass` keeps the key out of the notebook output so you don't accidentally commit it.

In [ ]:
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Groq API key (input hidden): ")
print("Key set — you won't need to re-enter it for the rest of this notebook.")

---
## 2 · How the pipeline works

Before running anything, here's the decision tree:

```
User pastes a docs URL
        │
        ▼
Does the URL end in .json/.yaml?  ──YES──▶  Fetch it directly as an OpenAPI spec
        │ NO                                          │
        ▼                                             │
Probe common spec locations                           │
(/openapi.json, /swagger.json …)                      │
        │                                             │
Found?──YES──▶  Parse OpenAPI spec ◀──────────────────┘
        │ NO         │
        ▼            ▼
Scrape docs     [Path A — fast, exact]
text with
BeautifulSoup
        │
        ▼
Send to Groq LLM → extract structured JSON
        │
        ▼
   [Path B — flexible, handles any docs]
        │
        └──────────────┐
                       ▼
         Check PyPI / npm for official SDK
                       │
                       ▼
         Generate wrapper class (Python or JS)
```

Both paths produce the same dict shape: `{base_url, auth_type, auth_details, endpoints, source}`. The code generator never needs to know which path ran.

---
## 3 · Demo A — JSONPlaceholder (LLM extraction path)

JSONPlaceholder is a free fake REST API used for testing. It has no OpenAPI spec file — just human-readable docs. This is the harder case, and the more common one in the real world.

In [ ]:
%%time
from scraper.doc_scraper import scrape_docs

DOCS_URL = "https://jsonplaceholder.typicode.com/guide/"

scraped = scrape_docs(DOCS_URL, max_pages=3)
text    = scraped.get("combined_text", "")

print(f"Pages fetched : {len(scraped.get('pages', []))}")
print(f"Total chars   : {len(text):,}")
print(f"\nSample (first 400 chars after noise removal):")
print("-" * 60)
print(text[:400])

The scraper strips `<nav>`, `<script>`, `<footer>` and other noise tags before extracting text, so what goes to the LLM is clean prose — not a wall of HTML.

In [ ]:
%%time
from llm.extractor import extract

# No spec file exists for this URL, so the extractor falls back to Groq LLM.
# The LLM reads the prose and returns structured JSON.
spec = extract(DOCS_URL, text)

print(f"Extraction path : {spec['source']}")
print(f"Base URL        : {spec['base_url']}")
print(f"Auth type       : {spec['auth_type']}")
print(f"Endpoints found : {len(spec['endpoints'])}")
print()
print(f"{'METHOD':<8} {'PATH':<40} PARAMETERS")
print("-" * 75)
for ep in spec["endpoints"]:
    params = ", ".join(p["name"] for p in ep.get("parameters", [])) or "—"
    print(f"{ep['method']:<8} {ep['path']:<40} {params}")

### 3a · SDK check

Before generating a custom wrapper, api-forge checks whether an official SDK already exists on PyPI (Python) or npm (JavaScript). No point reinventing the wheel.

In [ ]:
from llm.sdk_checker import check as check_sdk

sdk = check_sdk(DOCS_URL, language="python")

if sdk:
    print(f"⚠️  Official SDK found: {sdk['name']} v{sdk['version']}")
    print(f"   Install : {sdk['install_cmd']}")
    print(f"   URL     : {sdk['url']}")
    print()
    print("   The generated wrapper is still useful for zero-dependency setups.")
else:
    # JSONPlaceholder is a testing tool, not a real product — no SDK expected
    print("No official SDK found on PyPI — generating custom wrapper.")

### 3b · Generate the Python wrapper

In [ ]:
%%time
from generator.code_generator import generate

python_code = generate(spec, language="python")

print(f"Generated {len(python_code.splitlines())} lines of Python.")
print()
# Show the first 60 lines — enough to see the class definition and auth handling
print("\n".join(python_code.splitlines()[:60]))
print("\n... (truncated for preview — full class saved below)")

In [ ]:
# Save to disk so we can import and actually run it
with open("/tmp/jsonplaceholder_client.py", "w") as f:
    f.write(python_code)

print("Saved to /tmp/jsonplaceholder_client.py")
print("Let's import it and make a real API call with it.")

### 3c · Live test — actually run the generated code

This is the proof. We import the generated file and call a real endpoint with it.

In [ ]:
import importlib.util, sys

# Dynamically import the generated wrapper
spec_obj = importlib.util.spec_from_file_location(
    "jsonplaceholder_client", "/tmp/jsonplaceholder_client.py"
)
module = importlib.util.module_from_spec(spec_obj)
spec_obj.loader.exec_module(module)

# Find the client class (it'll be the first class defined in the file)
import inspect
client_class = next(
    obj for _, obj in inspect.getmembers(module, inspect.isclass)
    if obj.__module__ == module.__name__
)

print(f"Imported class: {client_class.__name__}")

# Instantiate and call a real endpoint
client = client_class()

# Call GET /posts — should return 100 posts from the live API
posts = client.list_posts() if hasattr(client, "list_posts") else client._request("GET", "/posts")

print(f"\n✅ Got {len(posts)} posts from the live JSONPlaceholder API")
print(f"\nFirst post:")
print(json.dumps(posts[0], indent=2))

### 3d · Generate the JavaScript wrapper

Same spec, different output template. Uses `fetch` — no extra dependencies.

In [ ]:
%%time
js_code = generate(spec, language="javascript")

print(f"Generated {len(js_code.splitlines())} lines of JavaScript.")
print()
print("\n".join(js_code.splitlines()[:40]))
print("\n... (truncated)")

---
## 4 · Demo B — Petstore (OpenAPI spec path)

Now the interesting contrast. The Petstore API publishes a machine-readable OpenAPI spec at a known URL. api-forge detects this and parses it directly — **no LLM call needed for extraction** — so it's faster and the endpoint list is exact.

We pass the `.json` URL directly and the extractor short-circuits to the spec parser.

In [ ]:
%%time
PETSTORE_URL = "https://petstore3.swagger.io/api/v3/openapi.json"

# scraped_text is empty string here — the extractor won't need it
# because it detects the .json extension and fetches the spec directly
petstore_spec = extract(PETSTORE_URL, "")

print(f"Extraction path : {petstore_spec['source']}   ← no LLM involved")
print(f"Base URL        : {petstore_spec['base_url']}")
print(f"Auth type       : {petstore_spec['auth_type']}")
print(f"Endpoints found : {len(petstore_spec['endpoints'])}")
print()
print(f"{'METHOD':<8} {'PATH':<45} DESCRIPTION")
print("-" * 90)
for ep in petstore_spec["endpoints"]:
    desc = (ep.get("description") or "")[:40]
    print(f"{ep['method']:<8} {ep['path']:<45} {desc}")

In [ ]:
%%time
petstore_code = generate(petstore_spec, language="python")

print(f"Generated {len(petstore_code.splitlines())} lines for the Petstore API")
print()
print("\n".join(petstore_code.splitlines()[:50]))
print("\n... (truncated)")

---
## 5 · Side-by-side comparison

Both paths produce the same shape. Here's a quick comparison of what came back.

In [ ]:
rows = [
    ("API",              "JSONPlaceholder",            "Petstore (Swagger)"),
    ("Docs URL",         DOCS_URL[:45] + "…",          PETSTORE_URL[:45] + "…"),
    ("Extraction path",  spec['source'],                petstore_spec['source']),
    ("LLM call needed?", "Yes (Path B)",                "No  (Path A)"),
    ("Base URL",         spec['base_url'],               petstore_spec['base_url']),
    ("Auth type",        spec['auth_type'],              petstore_spec['auth_type']),
    ("Endpoints",        str(len(spec['endpoints'])),    str(len(petstore_spec['endpoints']))),
    ("Python lines",     str(len(python_code.splitlines())), str(len(petstore_code.splitlines()))),
]

col_w = [22, 35, 35]
sep   = "+" + "+".join("-" * (w + 2) for w in col_w) + "+"

print(sep)
for i, row in enumerate(rows):
    line = "| " + " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row)) + " |"
    print(line)
    if i == 0:
        print(sep)
print(sep)

---
## 6 · Use-case filtering (large APIs)

For APIs with many endpoints, you can describe what you're building and the tool filters down to only the relevant ones. This avoids generating a 200-method class for a Stripe integration when you only need payments.

In [ ]:
# Petstore has 15+ endpoints — let's say we only care about pets, not users or orders
USE_CASE = "find available pets and upload photos of new animals"

filtered_spec = extract(PETSTORE_URL, "", use_case=USE_CASE)

print(f"Total endpoints in spec : {len(petstore_spec['endpoints'])}")
print(f"After filtering for     : '{USE_CASE}'")
print(f"Relevant endpoints kept : {len(filtered_spec['endpoints'])}")
print()
for ep in filtered_spec["endpoints"]:
    print(f"  {ep['method']:<7} {ep['path']}")

---
## 7 · Summary

What just ran, end-to-end:

In [ ]:
summary = [
    ("✅", "Scraper",       "Fetched + cleaned prose docs from a live URL"),
    ("✅", "Path B",        "Groq LLM read prose docs → returned structured JSON spec"),
    ("✅", "Path A",        "Parsed OpenAPI spec directly — no LLM needed for extraction"),
    ("✅", "SDK check",     "Searched PyPI and npm for an official SDK before generating"),
    ("✅", "Python gen",    f"Generated {len(python_code.splitlines())} lines of typed, documented Python"),
    ("✅", "JS gen",        f"Generated {len(js_code.splitlines())} lines of async JavaScript (fetch, no deps)"),
    ("✅", "Live test",     "Imported the generated file and called the real API — got real data"),
    ("✅", "Use-case filter", "Filtered 15 endpoints down to the relevant subset for a given task"),
]

for icon, label, desc in summary:
    print(f"{icon}  {label:<18} {desc}")

---
## 8 · Run the full Streamlit UI

The notebook shows each step in isolation. The Streamlit app ties them together with a UI — paste a URL, pick a language, click Generate.

```bash
# From inside the api-forge folder:
pip install streamlit
streamlit run app.py
```

The sidebar has a Groq key input field, so no `.env` file is needed to run it.

---

> **Repo:** https://github.com/ajsalsameer/api-forge  
> **Built for:** Claysys Hackathon — Smart DevTool for API Integration